In [9]:
import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv("../.env")
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
class CricbuzzExplorer:
    """Calls the Cricbuzz API, saves raw JSON, and reports remaining quota."""

    def __init__(self):
        self.base_url = f"https://{os.getenv('RAPIDAPI_HOST')}"
        self.headers = {
            "X-RapidAPI-Key": os.getenv("RAPIDAPI_KEY"),
            "X-RapidAPI-Host": os.getenv("RAPIDAPI_HOST"),
        }

    def fetch(self, endpoint: str, save_as: str, params: dict | None = None) -> dict | None:
        resp = requests.get(self.base_url + endpoint, headers=self.headers, params=params, timeout=15)
        remaining = resp.headers.get("x-ratelimit-requests-remaining", "unknown")
        print(f"{endpoint} -> {resp.status_code} | quota remaining: {remaining}")
        if resp.status_code != 200:
            print(resp.text[:300])
            return None
        data = resp.json()
        (RAW_DIR / f"{save_as}.json").write_text(json.dumps(data, indent=2), encoding="utf-8")
        return data


api = CricbuzzExplorer()

In [11]:
live = api.fetch("/matches/v1/live", "matches_live")
recent = api.fetch("/matches/v1/recent", "matches_recent")
upcoming = api.fetch("/matches/v1/upcoming", "matches_upcoming")

/matches/v1/live -> 200 | quota remaining: 196
/matches/v1/recent -> 200 | quota remaining: 195
/matches/v1/upcoming -> 200 | quota remaining: 194


In [12]:
type_match = recent["typeMatches"][0]
series = type_match["seriesMatches"][0]["seriesAdWrapper"]
match_info = series["matches"][0]["matchInfo"]
print(json.dumps(match_info, indent=2)[:1500])
MATCH_ID = match_info["matchId"]

{
  "matchId": 161190,
  "seriesId": 12543,
  "seriesName": "India tour of New Zealand 2026",
  "matchDesc": "1st T20I",
  "matchFormat": "T20",
  "startDate": "1792652400000",
  "endDate": "1792665000000",
  "state": "Complete",
  "status": "India won by 4 wkts",
  "team1": {
    "teamId": 13,
    "teamName": "New Zealand",
    "teamSName": "NZ",
    "imageId": 776333
  },
  "team2": {
    "teamId": 2,
    "teamName": "India",
    "teamSName": "IND",
    "imageId": 776162
  },
  "venueInfo": {
    "id": 371,
    "ground": "Hagley Oval",
    "city": "Christchurch",
    "timezone": "+13:00",
    "latitude": "-43.535831",
    "longitude": "172.624863"
  },
  "currBatTeamId": 2,
  "seriesStartDt": "1792627200000",
  "seriesEndDt": "1796256000000",
  "isTimeAnnounced": true,
  "stateTitle": "Complete",
  "isFantasyEnabled": true,
  "hideScoreStatus": "Match completed"
}


In [13]:
match_center = api.fetch(f"/mcenter/v1/{MATCH_ID}", f"match_{MATCH_ID}_info")
scorecard = api.fetch(f"/mcenter/v1/{MATCH_ID}/hscard", f"match_{MATCH_ID}_scorecard")

/mcenter/v1/161190 -> 200 | quota remaining: 193
/mcenter/v1/161190/hscard -> 200 | quota remaining: 192


In [14]:
search = api.fetch("/stats/v1/player/search", "player_search_kohli", params={"plrN": "Kohli"})
PLAYER_ID = search["player"][0]["id"]
player_info = api.fetch(f"/stats/v1/player/{PLAYER_ID}", f"player_{PLAYER_ID}_info")
player_bat = api.fetch(f"/stats/v1/player/{PLAYER_ID}/batting", f"player_{PLAYER_ID}_batting")
player_bowl = api.fetch(f"/stats/v1/player/{PLAYER_ID}/bowling", f"player_{PLAYER_ID}_bowling")

/stats/v1/player/search -> 200 | quota remaining: 191
/stats/v1/player/1413 -> 200 | quota remaining: 190
/stats/v1/player/1413/batting -> 200 | quota remaining: 189
/stats/v1/player/1413/bowling -> 200 | quota remaining: 188


In [15]:
series_list = api.fetch("/series/v1/international", "series_international")
teams = api.fetch("/teams/v1/international", "teams_international")
top_stats = api.fetch("/stats/v1/topstats/0", "topstats_most_runs", params={"statsType": "mostRuns"})
rankings = api.fetch("/stats/v1/rankings/batsmen", "rankings_odi_batsmen", params={"formatType": "odi"})

/series/v1/international -> 200 | quota remaining: 187
/teams/v1/international -> 200 | quota remaining: 186
/stats/v1/topstats/0 -> 200 | quota remaining: 185
/stats/v1/rankings/batsmen -> 200 | quota remaining: 184


In [16]:
VENUE_ID = match_info["venueInfo"]["id"]
venue = api.fetch(f"/venues/v1/{VENUE_ID}", f"venue_{VENUE_ID}")

/venues/v1/371 -> 200 | quota remaining: 183


In [17]:
def show_keys(obj, depth=0, max_depth=3):
    """Print the nested key structure of a JSON object (first list item only)."""
    pad = "  " * depth
    if depth > max_depth:
        return
    if isinstance(obj, dict):
        for k, v in obj.items():
            preview = "" if isinstance(v, (dict, list)) else f" = {str(v)[:40]}"
            print(f"{pad}{k} ({type(v).__name__}){preview}")
            show_keys(v, depth + 1, max_depth)
    elif isinstance(obj, list) and obj:
        print(f"{pad}[list of {len(obj)}]")
        show_keys(obj[0], depth + 1, max_depth)


def find_keys(obj, words, path=""):
    """Find keys containing any of the given words anywhere in the JSON."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{path}.{k}"
            if any(w in k.lower() for w in words):
                print(f"{p} = {str(v)[:80]}")
            find_keys(v, words, p)
    elif isinstance(obj, list) and obj:
        find_keys(obj[0], words, f"{path}[0]")

In [18]:
find_keys(match_center, ["toss", "result", "win", "margin", "decision"])

.tossstatus = India opt to bowl


In [19]:
show_keys(scorecard, max_depth=4)

scorecard (list)
  [list of 2]
    inningsid (int) = 1
    batsman (list)
      [list of 11]
        id (int) = 9443
        balls (int) = 17
        runs (int) = 24
        fours (int) = 3
        sixes (int) = 1
        strkrate (str) = 141.18
        name (str) = Tim Seifert
        nickname (str) = Tim Seifert
        iscaptain (bool) = False
        iskeeper (bool) = True
        outdec (str) = c Sanju Samson b Mayank Yadav
        videotype (str) = 
        videourl (str) = 
        videoid (int) = 0
        planid (int) = 0
        imageid (int) = 0
        premiumvideourl (str) = 
        iscbplusfree (bool) = False
        ispremiumfree (bool) = False
        inmatchchange (str) = 
        isoverseas (bool) = False
        playingxichange (str) = 
    bowler (list)
      [list of 5]
        id (int) = 1726
        overs (str) = 4
        maidens (int) = 0
        wickets (int) = 0
        runs (int) = 49
        economy (str) = 12.2
        name (str) = Bhuvneshwar Kumar
     

In [20]:
print("=== VENUE ===")
print(json.dumps(venue, indent=2)[:800])
print("\n=== PLAYER INFO ===")
find_keys(player_info, ["role", "bat", "bowl", "team", "name"])
print("\n=== PLAYER BATTING ===")
show_keys(player_bat, max_depth=3)

=== VENUE ===
{
  "ground": "Hagley Oval",
  "city": "Christchurch",
  "country": "New Zealand",
  "timezone": "+13:00",
  "established": 1886,
  "capacity": "20,000",
  "knownAs": "Hagley Oval",
  "homeTeam": "Canterbury",
  "curator": "Rupert Bool",
  "profile": "<b>Venue description:</b>\n\nChristchurch waited almost three years since 2011 for international cricket to return to the city. Most parts of the city were severely damaged by the earthquake in February 2011, including Lancaster Park, previously the venue for international cricket in Christchurch. The last game played here was an ODI that involved New Zealand and Pakistan. Later in 2013, Canterbury Cricket got the approval from the Environment Court to develop Hagley Oval as an international venue. \n\nHagley Oval is situated in the central c

=== PLAYER INFO ===
.bat = Right Handed Bat
.bowl = Right-arm medium
.name = Virat Kohli
.fullName = Virat Kohli
.role = Batsman
.intlTeam = India
.teams = India, Delhi, India Red, Ind

In [21]:
import pandas as pd
print(pd.to_datetime(int(match_info["startDate"]), unit="ms", utc=True).tz_convert("Asia/Kolkata"))

2026-10-22 12:30:00+05:30
